# GaussLaw — 3D projection of a non-solenoidal magnetic field

`MagneticStandalone3D` · opt-in Gauss-law correction. This is the 3D extrusion twin of `MagneticStandalone/GaussLaw/`.

The full vector field starts from `b=(x*(1-x), 0, 0)`, whose reported Green–Gauss divergence is nonzero locally but whose net boundary flux is compatible. The projection is checked at interior probes on the coarse/standard/fine mesh ladder.

In [ ]:
from trustutils import run
run.TRUST_parameters("1.9.8")
run.reset()


## Run the mesh-convergence ladder

All levels use `src/gauss_law.data`. The active x direction follows the shared 16/32/64-cell ladder; y and z remain coarse because the solution is uniform in both. Three cells in each transverse direction avoid a parity-sensitive null mode in this 3D cell-centred operator. `Polyedriser dom` converts the Pave mesh to the polyhedra required by PolyMAC_MPFA. This 3D twin runs sequentially; the 2D form separately exercises the MPI path.

In [ ]:
import os
import numpy as np
from trustutils.files import SonFile

BUILD = run.BUILD_DIRECTORY
run.reset()
NX0 = 17
meshes = {
    "coarse":   {"nx": NX0,     "ny": 4, "nz": 4},
    "standard": {"nx": 2*NX0-1, "ny": 4, "nz": 4},
    "fine":     {"nx": 4*NX0-3, "ny": 4, "nz": 4},
}
cases = {}
for level, mesh in meshes.items():
    cases[level] = run.addCaseFromTemplate("gauss_law.data", level, dic=mesh,
                                           nbProcs=1, excluNR=True)
run.printCases()
run.runCases()


## Gauss-law checks

As in 2D, TRUST records the t=0 probes before the initial projection hook. The assertions therefore use samples after the first time step. Each probe is interior, away from the boundary-cell Green–Gauss approximation.

In [ ]:
def read_probe(level, tag):
    case_name = cases[level].dataFileName_
    candidates = (f"{BUILD}/{level}/{case_name}_{tag}.son",
                  f"{BUILD}/{case_name}_{tag}.son")
    path = next((candidate for candidate in candidates if os.path.exists(candidate)), None)
    assert path is not None, f"missing probe file: {candidates}"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"empty probe file: {path}"
    t, values = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(values, dtype=float)

TAGS = ("SONDE_DIV_1", "SONDE_DIV_2", "SONDE_DIV_3")
for level in ("coarse", "standard", "fine"):
    worst = 0.0
    for tag in TAGS:
        t, values = read_probe(level, tag)
        after_initial_projection = t > 0.
        assert np.any(after_initial_projection), f"no post-initialization samples for {level}/{tag}"
        worst = max(worst, float(np.max(np.abs(values[after_initial_projection]))))
    print(f"{level:>8}: max |divergence_magnetique| = {worst:.4e}")
    assert worst < 1.e-8, f"Gauss-law correction failed on {level}: {worst:.3e}"
print("OK: interior magnetic divergence remains at solver tolerance across the 3D mesh scan.")
